# Data Connect Python Library - Usage

***Version Note:***
This notebook corresponds to the **dataconnect-library-python** `v1.0.0`. Ensure that you are working with this specific version. 

***View version-matched documentation:***
On GitHub, use the branch/tag selector to switch to your version's tag (e.g., `v1.0.0`).

## Setup

**Prerequisites**
* Python 3.13
* pip3 (pip for Windows)

**Credentials**
* An iMedidata Account
* Access to **DataConnect**'s **Developer Center** and **Transformations** in iMedidata

*Note:* You will need a generated `User Authentication Token` from **Developer Center** to use this library.

In [ ]:
%pip install git+https://github.com/mdsol/dataconnect-library-python.git@v1.0.0

## Usage
It is recommended that the `User Authentication Token` generated from `Developer Center` is stored in a separate secrets file. When using functions without context, the connection has to be explicitly closed. Alternatively, a context manager (`with` statement, for example) can be used to ensure proper resource management.

### Get all available studies

Get all available studies or search for a study by name (without using context).

In [ ]:
from dataconnect import DataConnectClient

user_token = "usertoken"  # From iMedidata > Data Connect > Developer Center

dataconnect_client = DataConnectClient.connect(token=user_token)

try:
    study_result = dataconnect_client.get_studies(search_study_name="")

    if study_result:
        print(f"\n--- Total Studies: {study_result.total_records} ---\n")

        for study in study_result.studies:
            print(f"• Study: {study.name}")
            print("  Envs: ")

            if study.environments:
                for env in study.environments:
                    print(f"    - {env.uuid} : {env.name}")
            print("\n")
    else:
        print("No Studies found.")
except Exception as e:
    print(e)
finally:
    dataconnect_client.close()

Get all available studies or search for a study by name (using `with` for context management).

In [ ]:
from dataconnect import DataConnectClient

user_token = "usertoken"  # From iMedidata > Data Connect > Developer Center

try:
    with DataConnectClient.connect(token=user_token) as dataconnect_client:
        study_result = dataconnect_client.get_studies(search_study_name="")

        if study_result:
            print(f"\n--- Total Studies: {study_result.total_records} ---\n")

            for study in study_result.studies:
                print(f"• Study: {study.name}")
                print("  Envs: ")

                if study.environments:
                    for env in study.environments:
                        print(f"    - {env.uuid} : {env.name}")
                print("\n")

        else:
            print("No Studies found.")
except Exception as e:
    print(e)

## Work with datasets

### Search datasets in a study environment

Retrieve the study environment uuid from "Developer Info" in iMedidata Platform.

In [ ]:
from uuid import UUID

from dataconnect import DataConnectClient

try:
    user_token = "usertoken"  # From iMedidata > Data Connect > Developer Center
    search_name = "datasetname"  # complete dataset name or wildcard

    with DataConnectClient.connect(token=user_token) as dataconnect_client:
        study_env_uuid = UUID("<get_study_environment_uuid_from_iMedidata>")

        datasets = dataconnect_client.get_datasets(
            study_environment_uuid=study_env_uuid,
            search_dataset_name=search_name,
            page=1,
            page_size=5,
        )

        print(f"Total datasets available across all pages: {datasets.total_records}")
        print(
            f"Page: {datasets.pagination.page}, "
            f"Page size: {datasets.pagination.page_size}, "
            f"Total pages: {datasets.pagination.total_pages}"
        )

        for dataset in datasets.items:
            print(f"\n  Dataset: {dataset.dataset_name}")
            print(f"  UUID:    {dataset.dataset_uuid}")

except Exception as e:
    print(e)

### Retrieve all dataset versions.

In [ ]:
from uuid import UUID

from dataconnect import DataConnectClient

user_token = "usertoken"  # From iMedidata > Data Connect > Developer Center

try:
    with DataConnectClient.connect(token=user_token) as dataconnect_client:
        dataset_uuid = UUID("<dataset_uuid_from_get_datasets()_call>")

        dataset_versions = dataconnect_client.get_dataset_versions(dataset_uuid=dataset_uuid)

        for version in dataset_versions:
            print(f"  Dataset: {version.dataset_name}")
            print(f"  Version: {version.dataset_version}")
            print(f"  Dataset Version UUID:    {version.dataset_uuid}\n")

except Exception as e:
    print(e)

### Fetch records of a specific dataset
Data fetched based on a `dataset_uuid` are in pandas `DataFrame` format. You can use the optional parameter `first_n_rows` to limit the number of rows fetched, which is useful for large datasets.

In [ ]:
from uuid import UUID

from dataconnect import DataConnectClient

user_token = "usertoken"  # From iMedidata > Data Connect > Developer Center

try:
    with DataConnectClient.connect(token=user_token) as dataconnect_client:
        dataset_uuid = UUID("<dataset_uuid_from_get_datasets()_call>")

        # Fetch only the first 10 rows from the server
        data = dataconnect_client.fetch_data(dataset_uuid=dataset_uuid, first_n_rows=10)

        print(data)

except Exception as e:
    print(e)

## Dry Publish and Publish
Along with the `User Authentication Token` as mentioned above, a `Project Token` is required to publish datasets to a `Data Connect` Transformation project. 

You can retrieve this by navigating to `iMedidata` > `Data Connect` > `Transformations` and by clicking on the kebab menu on a project row and selecting `View/Modify`.

### Discover supported datetime formats

Use `get_datetime_formats` to list the format patterns the server accepts for the `datetime_formats` argument of `dry_publish` / `publish`. Pass `format_type="date"` or `format_type="datetime"` to restrict the result; the default `"all"` returns both kinds.

In [ ]:
from dataconnect import DataConnectClient

user_token = "usertoken"  # From iMedidata > Data Connect > Developer Center
project_token = "project_token"  # From iMedidata > Data Connect > Transformations

try:
    with DataConnectClient.connect(token=user_token) as dataconnect_client:
        # Default: return both date and datetime formats
        formats_result = dataconnect_client.get_datetime_formats(
            project_token=project_token,
            format_type="all",
        )

        print("All supported formats:")
        for fmt in formats_result.all():
            print(f"  {fmt.format}  [{fmt.type}]")

        print("\nDate-only formats:   ", formats_result.dates())
        print("Datetime formats:    ", formats_result.datetimes())

except Exception as e:
    print(e)

### Dry Publish (validate without persisting)

Before publishing, use `dry_publish` to validate your dataset against the server without committing any changes. This lets you catch errors early.

In [ ]:
from uuid import UUID

import pandas as pd

from dataconnect import DataConnectClient

user_token = "usertoken"  # From iMedidata > Data Connect > Developer Center
project_token = "project_token"  # From iMedidata > Data Connect > Transformations

# Configuration
dataset_name = "my_new_dataset"
key_columns: list[str] = ["patient_id", "site_id"]
source_datasets = [
    UUID("f30a8619-8348-330b-8a66-34d837303a8c")
]  # Any number of source datasets, in UUID format and comma-separated as a list
datetime_formats = {"visit_date": "yyyy-MM-dd"}

# Data to Publish - pandas DataFrame
data = pd.DataFrame(
    {
        "VISIT_NAME": ["Adverse Events", "Adverse Events", "Adverse Events"],
        "SITE_ID": ["1010", "1042", "1001"],
        "PATIENT_ID": ["10101097", "10421089", "10011029"],
        "VISIT_DATE": pd.to_datetime(["2020-02-01", "2021-02-01", "2022-02-01"]),
    }
)

try:
    with DataConnectClient.connect(token=user_token) as dataconnect_client:
        dry_publish_result = dataconnect_client.dry_publish(
            project_token=project_token,
            dataset_name=dataset_name,
            key_columns=key_columns,
            source_datasets=source_datasets,
            datetime_formats=datetime_formats,
            data=data,
        )  # pandas DataFrame to publish

        print(f"Dry publish result: 	  {dry_publish_result.status}")
        print(f"Schema valid:             {dry_publish_result.is_schema_valid}")
        print(f"Config valid:             {dry_publish_result.is_config_valid}")
        print(f"Dataset valid:            {dry_publish_result.is_dataset_valid}")
        print(f"Dataset name:             {dry_publish_result.dataset_name}")
        print(f"Dataset version:          {dry_publish_result.dataset_version}")
        print(f"Number of columns:        {dry_publish_result.no_of_columns}")
        print(f"Valid record count:       {dry_publish_result.valid_record_count}")
        print(f"Duplicate record count:   {dry_publish_result.duplicate_record_count}")
        print(f"Invalid record count:     {dry_publish_result.invalid_record_count}")

        if dry_publish_result.errors:
            print(f"\nErrors: {dry_publish_result.errors}")

        if dry_publish_result.invalid_datetime_formats:
            print(f"\nInvalid datetime formats: {dry_publish_result.invalid_datetime_formats}")

        if dry_publish_result.invalid_records is not None and not dry_publish_result.invalid_records.empty:
            print("\nInvalid records:")
            print(dry_publish_result.invalid_records)

except Exception as e:
    print(e)

### Publish (persist the dataset)

Once the dry publish validates successfully, publish the dataset to the server. The now published dataset should be visible in `iMedidata` > `Data Connect` > `Datasets` within a few minutes. 

In [ ]:
from dataconnect import DataConnectClient

user_token = "usertoken"  # From iMedidata > Data Connect > Developer Center
project_token = "project_token"  # From iMedidata > Data Connect > Transformations

# Configuration
dataset_name = "my_new_dataset"
key_columns: list[str] = ["patient_id", "site_id"]
source_datasets = [
    UUID("f30a8619-8348-330b-8a66-34d837303a8c")
]  # Any number of source datasets, in UUID format and comma-separated as a list
datetime_formats = {"visit_date": "yyyy-MM-dd"}

# Data to Publish - pandas DataFrame
data = pd.DataFrame(
    {
        "VISIT_NAME": ["Adverse Events", "Adverse Events", "Adverse Events"],
        "SITE_ID": ["1010", "1042", "1001"],
        "PATIENT_ID": ["10101097", "10421089", "10011029"],
        "VISIT_DATE": pd.to_datetime(["2020-02-01", "2021-02-01", "2022-02-01"]),
    }
)

try:
    with DataConnectClient.connect(token=user_token) as dataconnect_client:
        publish_result = dataconnect_client.publish(
            project_token=project_token,
            dataset_name=dataset_name,
            key_columns=key_columns,
            source_datasets=source_datasets,
            datetime_formats=datetime_formats,
            data=data,
        )  # pandas DataFrame to publish

        print(f"Publish status:           {publish_result.status}")
        print(f"Dataset name:             {publish_result.dataset_name}")
        print(f"Dataset UUID:             {publish_result.dataset_uuid}")
        print(f"Dataset version:          {publish_result.dataset_version}")
        print(f"Dataset batch number:     {publish_result.dataset_batch_number}")
        print(f"Valid record count:       {publish_result.valid_record_count}")
        print(f"Duplicate record count:   {publish_result.duplicate_record_count}")
        print(f"Invalid record count:     {publish_result.invalid_record_count}")

        if publish_result.invalid_records is not None and not publish_result.invalid_records.empty:
            print("\nInvalid records:")
            print(publish_result.invalid_records)

except Exception as e:
    print(e)